# 05. ASR Experiments: Whisper-tiny int8 (± Silero VAD)

In [1]:
import json
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download

from meme_organizer.config import MLFLOW_EXPERIMENT_ASR
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.extraction.asr import AsrExperimentAdapter, SherpaOnnxAsrExtractor
from meme_organizer.extraction.audio import AudioExtractor

ROOT = Path("..").resolve()
GOLD_PATH = ROOT / "data" / "gold" / "gold_dataset.jsonl"
SILVER_DIR = ROOT / "data" / "silver"
records = [json.loads(line) for line in GOLD_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
vid_records = [r for r in records if r["media_type"] != "image" and r.get("transcript_or_text", "").strip()]
print(f"Video/GIF records with transcript_or_text: {len(vid_records)}")

repo = "csukuangfj/sherpa-onnx-whisper-tiny"
enc_path = Path(hf_hub_download(repo, "tiny-encoder.int8.onnx"))
dec_path = Path(hf_hub_download(repo, "tiny-decoder.int8.onnx"))
tok_path = Path(hf_hub_download(repo, "tiny-tokens.txt"))
vad_path = Path(hf_hub_download("onnx-community/silero-vad", "onnx/model.onnx"))
results = []

2026/11/27 20:22:44 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Video/GIF records with transcript_or_text: 115


In [2]:
extractor_no_vad = SherpaOnnxAsrExtractor(
    model_type="whisper",
    encoder_path=enc_path,
    decoder_path=dec_path,
    tokens_path=tok_path,
    language="ru",
)
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_ASR) as exp:
    adapter = AsrExperimentAdapter(
        run_name="vid_sherpa_whisper_tiny_int8",
        extractor=extractor_no_vad,
        audio_extractor=AudioExtractor(),
        records=vid_records,
        silver_dir=SILVER_DIR,
        model_artifacts={"encoder": enc_path, "decoder": dec_path},
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "whisper_tiny_no_vad", **res["quality"], **res["performance"]})
print("No VAD:", res["quality"], res["performance"])

/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (90 tokens for 15.02 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-recognizer-whisper-impl.h:DecodeStream:131 Only waves less than 30 seconds are supported. We process only the first 30 seconds and discard the remaining data


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (10 tokens for 1.79 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (87 tokens for 14.58 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (152 tokens for 25.43 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (53 tokens for 8.96 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


... (18 outputs left out)

/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (75 tokens for 12.63 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-recognizer-whisper-impl.h:DecodeStream:131 Only waves less than 30 seconds are supported. We process only the first 30 seconds and discard the remaining data


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (85 tokens for 14.21 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


🏃 View run vid_sherpa_whisper_tiny_int8 at: http://localhost:5000/#/experiments/14/runs/0ab32483501148e8b6aa54217c4158c8
🧪 View experiment at: http://localhost:5000/#/experiments/14
No VAD: {'cer': 0.9489, 'wer': 1.2664, 'empty_rate': 0.0174} {'audio_extract_p50_ms': 39.87, 'inference_p50_ms': 274.83, 'inference_p95_ms': 1203.53, 'total_p50_ms': 328.25, 'total_p95_ms': 1265.88, 'throughput_items_per_sec': 2.11, 'model_size_mb': 98.03, 'total_pipeline_time_sec': 54.549}


In [3]:
extractor_vad = SherpaOnnxAsrExtractor(
    model_type="whisper",
    encoder_path=enc_path,
    decoder_path=dec_path,
    tokens_path=tok_path,
    vad_model_path=vad_path,
    language="ru",
)
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_ASR) as exp:
    adapter = AsrExperimentAdapter(
        run_name="vid_sherpa_whisper_tiny_int8_vad",
        extractor=extractor_vad,
        audio_extractor=AudioExtractor(),
        records=vid_records,
        silver_dir=SILVER_DIR,
        model_artifacts={"encoder": enc_path, "decoder": dec_path, "vad": vad_path},
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "whisper_tiny_silero_vad", **res["quality"], **res["performance"]})
print("With VAD:", res["quality"], res["performance"])
print(
    pd.DataFrame(results)[
        ["run", "cer", "wer", "empty_rate", "audio_extract_p50_ms", "inference_p50_ms", "total_p50_ms", "model_size_mb"]
    ].to_string(index=False)
)

/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (1 tokens for 0.33 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (1 tokens for 0.33 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.
/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.34 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (1 tokens for 0.33 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.
/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.34 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (1 tokens for 0.33 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (1 tokens for 0.33 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.35 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.
/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.36 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


... (43 outputs left out)

/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.34 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.
/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.35 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.35 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.
/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.34 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (2 tokens for 0.36 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.
/project/sherpa-onnx/csrc/offline-whisper-greedy-search-decoder.cc:Decode:294 Warning: the decode stopped at the token limit (1 tokens for 0.33 s of audio, max_tokens_per_second=6) rather than at an end-of-text token. The text may be truncated. Set the per-stream option 'max_tokens_per_second' to a larger value to allow more tokens.


🏃 View run vid_sherpa_whisper_tiny_int8_vad at: http://localhost:5000/#/experiments/14/runs/53efeb4c819640babc4bb7873ee49e34
🧪 View experiment at: http://localhost:5000/#/experiments/14
With VAD: {'cer': 0.9875, 'wer': 0.9982, 'empty_rate': 0.1478} {'audio_extract_p50_ms': 49.26, 'inference_p50_ms': 101.89, 'inference_p95_ms': 179.18, 'total_p50_ms': 151.4, 'total_p95_ms': 260.14, 'throughput_items_per_sec': 6.11, 'model_size_mb': 100.17, 'total_pipeline_time_sec': 18.957}
                    run    cer    wer  empty_rate  audio_extract_p50_ms  inference_p50_ms  total_p50_ms  model_size_mb
    whisper_tiny_no_vad 0.9489 1.2664      0.0174                 39.87            274.83        328.25          98.03
whisper_tiny_silero_vad 0.9875 0.9982      0.1478                 49.26            101.89        151.40         100.17


In [4]:
giga_repo = "csukuangfj/sherpa-onnx-nemo-ctc-giga-am-russian-2024-10-24"
giga_model = Path(hf_hub_download(giga_repo, "model.int8.onnx"))
giga_tokens = Path(hf_hub_download(giga_repo, "tokens.txt"))

for use_vad, rname in [(False, "vid_sherpa_gigaam_ctc_int8"), (True, "vid_sherpa_gigaam_ctc_int8_vad")]:
    ext = SherpaOnnxAsrExtractor(
        model_type="nemo_ctc",
        nemo_ctc_path=giga_model,
        tokens_path=giga_tokens,
        vad_model_path=vad_path if use_vad else None,
    )
    arts = {"nemo_ctc": giga_model, "tokens": giga_tokens}
    if use_vad:
        arts["vad"] = vad_path
    with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_ASR) as exp:
        adapter = AsrExperimentAdapter(
            run_name=rname,
            extractor=ext,
            audio_extractor=AudioExtractor(),
            records=vid_records,
            silver_dir=SILVER_DIR,
            model_artifacts=arts,
        )
        res = exp.create_pipeline(adapter).run()
    results.append({"run": rname, **res["quality"], **res["performance"]})
print(
    pd.DataFrame(results)[
        ["run", "cer", "wer", "empty_rate", "audio_extract_p50_ms", "inference_p50_ms", "total_p50_ms", "model_size_mb"]
    ].to_string(index=False)
)

model.int8.onnx: reconstructing file:   0%|          |  0.00B /  275MB            

model.int8.onnx: downloading bytes:           |  0.00B            

tokens.txt:   0%|          | 0.00/196 [00:00<?, ?B/s]

🏃 View run vid_sherpa_gigaam_ctc_int8 at: http://localhost:5000/#/experiments/14/runs/89597e67b6174705a9778a43e0bdcd4c
🧪 View experiment at: http://localhost:5000/#/experiments/14


/project/sherpa-onnx/csrc/circular-buffer.cc:Push:118 Overflow! n: 512, size: 960000, n+size: 960512, capacity: 960000. Increase capacity to: 1920000. (Original data is copied. No data loss!)
/project/sherpa-onnx/csrc/circular-buffer.cc:Push:118 Overflow! n: 512, size: 1920000, n+size: 1920512, capacity: 1920000. Increase capacity to: 3840000. (Original data is copied. No data loss!)


🏃 View run vid_sherpa_gigaam_ctc_int8_vad at: http://localhost:5000/#/experiments/14/runs/b57bfb24b8f24596a130e8dbab371a69
🧪 View experiment at: http://localhost:5000/#/experiments/14
                           run    cer    wer  empty_rate  audio_extract_p50_ms  inference_p50_ms  total_p50_ms  model_size_mb
           whisper_tiny_no_vad 0.9489 1.2664      0.0174                 39.87            274.83        328.25          98.03
       whisper_tiny_silero_vad 0.9875 0.9982      0.1478                 49.26            101.89        151.40         100.17
    vid_sherpa_gigaam_ctc_int8 0.6633 0.8969      0.2000                 36.80            645.39        683.42         262.08
vid_sherpa_gigaam_ctc_int8_vad 0.9957 1.0000      0.9043                 39.24             65.24        106.07         264.22
